# Unified Engine & Training Pipeline Playground
**Scope**: Unified HeteroGNN training, validation, and submission assembly  

This notebook covers:
1. **Data Pipeline Verification** — Load events, build unified graphs, inspect dimensions
2. **Model Smoke Test** — Forward pass through `UnifiedFloodModel`
3. **Loss Function Verification** — `FloodLoss` with variance-weighted SRMSE
4. **Training Pipeline** — Full training with curriculum learning
5. **Validation** — Leave-One-Event-Out with hierarchical SRMSE
6. **Training Curve Visualisation** — Loss, SRMSE, LR, TF ratio
7. **Submission Generation** — End-to-end inference pipeline

In [ ]:
# ── Cell 1: Imports & Setup ─────────────────────────────────────────
import sys
import os
import warnings
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Set data path (adjust if needed)
os.environ.setdefault("FLOOD_DATA_PATH", str(PROJECT_ROOT / "data"))

# Core imports
from src.config import RAW_DATA_PATH, MODELS_DIR, PROJECT_ROOT as PROJ_ROOT
from src.dataset import FloodDataset
from src.graph_builder_unified import (
    build_unified_graph,
    get_feature_dims,
    summarise_graph,
)
from src.model_unified import UnifiedFloodModel
from src.loss import (
    FloodLoss,
    standardized_rmse_loss,
    standardized_rmse_metric,
    SRMSEAccumulator,
    per_node_loss_breakdown,
)

# Device
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Project root : {PROJECT_ROOT}")
print(f"Data path    : {RAW_DATA_PATH}")
print(f"Device       : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")

## 1. Data Pipeline Verification

Load the dataset, verify static caching, and build a unified graph.

In [ ]:
# ── Cell 2: Load Dataset & Inspect ──────────────────────────────────
ds = FloodDataset(str(RAW_DATA_PATH), mode="train")
print(f"\nTotal events: {len(ds)}")
print(f"Models: {ds.get_model_ids()}")

for mid in ds.get_model_ids():
    events = ds.get_event_ids(mid)
    print(f"  Model_{mid}: {len(events)} events")

# Grab a sample event
sample = ds[0]
print(f"\nSample event: Model_{sample['model_id']}, Event_{sample['event_id']}")
print(f"  1D nodes static shape: {sample['static_1d_nodes'].shape}")
print(f"  2D nodes static shape: {sample['static_2d_nodes'].shape}")
print(f"  1D dynamic shape: {sample['dynamic_1d_nodes'].shape}")
print(f"  2D dynamic shape: {sample['dynamic_2d_nodes'].shape}")

# Verify static caching: load same model twice
_ = ds[0]  # First load
_ = ds[0]  # Should hit cache
print(f"\nStatic cache populated for models: {list(ds.static_cache.keys())}")

In [ ]:
# ── Cell 3: Build Unified Graph ─────────────────────────────────────
graph = build_unified_graph(sample)
print(summarise_graph(graph))

dims = get_feature_dims(graph)
print(f"\nFeature dimensions:")
for k, v in dims.items():
    print(f"  {k}: {v}")

## 2. Model Smoke Test

Instantiate the `UnifiedFloodModel` and run a forward pass.

In [ ]:
# ── Cell 4: Model Construction & Forward Pass ──────────────────────
model = UnifiedFloodModel.from_graph(graph, hidden_channels=64)
model = model.to(DEVICE)
print(model.summarise())

# Single-step forward pass
graph_dev = graph.to(DEVICE)
pred_1d, pred_2d, h_1d, h_2d = model.step(graph_dev, t=0)
print(f"Step output shapes:")
print(f"  pred_1d: {pred_1d.shape}")
print(f"  pred_2d: {pred_2d.shape}")
print(f"  h_1d[0]: {h_1d[0].shape}")

# Short rollout test (5 steps)
model.eval()
with torch.no_grad():
    preds_1d, preds_2d = model.rollout(
        graph_dev, spinup_steps=3, prediction_steps=5
    )
print(f"\nRollout output shapes:")
print(f"  preds_1d: {preds_1d.shape}")
print(f"  preds_2d: {preds_2d.shape}")

## 3. Loss Function Verification

Compute per-node stds and verify the `FloodLoss` module.

In [ ]:
# ── Cell 5: Node Stds & Loss ─────────────────────────────────────── 
# Compute per-node standard deviations (expensive — cache this!)
print("Computing per-node standard deviations...")
node_stds = ds.compute_node_stds(model_id="1")
stds_data = node_stds["1"]
stds_1d = torch.from_numpy(stds_data["1d"]).float()
stds_2d = torch.from_numpy(stds_data["2d"]).float()

print(f"  1D stds: shape={stds_1d.shape}, min={stds_1d.min():.4f}, "
      f"max={stds_1d.max():.4f}, mean={stds_1d.mean():.4f}")
print(f"  2D stds: shape={stds_2d.shape}, min={stds_2d.min():.4f}, "
      f"max={stds_2d.max():.4f}, mean={stds_2d.mean():.4f}")

# WARNING: nodes with very low std will dominate the loss
low_std_1d = (stds_1d < 0.01).sum().item()
low_std_2d = (stds_2d < 0.01).sum().item()
print(f"\n  Low-std (< 0.01) nodes: 1D={low_std_1d}, 2D={low_std_2d}")
print("  (These are 'always dry' nodes that need weight clamping)")

# Build FloodLoss
criterion = FloodLoss(
    node_stds_1d=stds_1d,
    node_stds_2d=stds_2d,
    clamp_weights=100.0,
    alpha=0.5,
    temporal_scheme="linear",
).to(DEVICE)

# Test loss computation
model.train()
with torch.no_grad():
    preds_1d_train, preds_2d_train = model.rollout(
        graph_dev, spinup_steps=3, prediction_steps=5
    )
    targets_1d = graph_dev["node_1d"].y[:8]
    targets_2d = graph_dev["node_2d"].y[:8]

    total_loss, breakdown = criterion.forward_combined(
        preds_1d_train[:8], targets_1d,
        preds_2d_train[:8], targets_2d,
    )

print(f"\nLoss test:")
print(f"  Total loss: {total_loss.item():.4f}")
print(f"  Breakdown: {breakdown}")

## 4. Training Pipeline

Run the full training pipeline with curriculum learning.

**Configuration notes**:
- Start with small `epochs` / `hidden_channels` for quick iteration
- Scale up once the pipeline is verified
- Monitor train loss & val SRMSE for convergence

In [ ]:
# ── Cell 6: Training with UnifiedTrainer ────────────────────────────
from src.trainer import TrainConfig, UnifiedTrainer

# Quick iteration config (scale up for real training)
cfg = TrainConfig(
    data_root=str(RAW_DATA_PATH),
    model_ids=["1"],            # Start with Model_1 only
    val_event_id="4",           # Leave-One-Event-Out
    
    # Architecture
    hidden_channels=64,
    num_gnn_layers=3,
    num_gru_layers=1,
    dropout=0.1,
    
    # Training
    epochs=5,                    # Start small for verification
    lr=1e-3,
    weight_decay=1e-5,
    grad_clip_norm=1.0,
    
    # Curriculum learning
    tf_warmup_epochs=2,
    tf_decay_epochs=2,
    tf_min_ratio=0.0,
    
    # Push-forward
    pushforward_K=5,             # Short window for quick test
    use_push_forward=True,
    temporal_scheme="linear",
    
    # Device
    device=str(DEVICE),
    use_amp=(DEVICE.type == "cuda"),
    
    # Output
    checkpoint_dir="checkpoints",
    log_dir="logs",
    verbose=True,
)

print("Training Config:")
for k, v in cfg.to_dict().items():
    print(f"  {k}: {v}")

In [ ]:
# ── Cell 7: Run Training ────────────────────────────────────────────
# WARNING: This is compute-intensive.
# For GPU training, increase epochs and hidden_channels.

trainer = UnifiedTrainer(cfg)
trainer.setup()
history = trainer.train()

## 5. Training Curve Visualisation

Plot the training loss, validation SRMSE, learning rate, and teacher forcing ratio over epochs.

In [ ]:
# ── Cell 8: Plot Training Curves ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Unified Engine Training Curves", fontsize=14, fontweight="bold")

epochs = range(len(history.train_loss))

# Training Loss
axes[0, 0].plot(epochs, history.train_loss, 'b-', linewidth=1.5, label="Train Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Training Loss (Weighted SRMSE Surrogate)")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Validation SRMSE
axes[0, 1].plot(epochs, history.val_srmse, 'r-', linewidth=1.5, label="Val SRMSE")
best_epoch = history.best_epoch
if best_epoch < len(history.val_srmse):
    best_idx = best_epoch - (epochs[0] if epochs else 0)
    if 0 <= best_idx < len(history.val_srmse):
        axes[0, 1].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5, label=f"Best (epoch {best_epoch})")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("SRMSE")
axes[0, 1].set_title(f"Validation SRMSE (Best: {history.best_val_srmse:.4f})")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Learning Rate
axes[1, 0].plot(epochs, history.lr_history, 'g-', linewidth=1.5)
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Learning Rate")
axes[1, 0].set_title("Learning Rate Schedule")
axes[1, 0].set_yscale("log")
axes[1, 0].grid(True, alpha=0.3)

# Teacher Forcing Ratio
ax_tf = axes[1, 1]
ax_tf.plot(epochs, history.tf_ratio, 'm-', linewidth=1.5, label="TF Ratio")
ax_tf.fill_between(epochs, 0, history.tf_ratio, alpha=0.1, color='m')
ax_tf.set_xlabel("Epoch")
ax_tf.set_ylabel("Teacher Forcing Ratio")
ax_tf.set_title("Curriculum Learning Schedule")
ax_tf.set_ylim(-0.05, 1.05)
ax_tf.axhline(y=0, color='k', linestyle=':', alpha=0.3)
ax_tf.legend()
ax_tf.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("logs/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(history.summary_str())

## 6. Validation with Full Competition Protocol

Run the `ValidationRunner` with exact competition scoring hierarchy.

In [ ]:
# ── Cell 9: Full Validation ─────────────────────────────────────────
from src.validate import ValidationRunner

# Load best model
try:
    trainer.load_best()
    print("Loaded best checkpoint.")
except FileNotFoundError:
    print("No best checkpoint found — using current model.")

runner = ValidationRunner(
    model=trainer.model,
    data_root=str(RAW_DATA_PATH),
    device=DEVICE,
    model_ids=["1"],
    spinup_steps=10,
    run_diagnostics=True,
    verbose=True,
)

val_result = runner.validate_holdout(val_event_id="4")
print(val_result.summary_str())

In [ ]:
# ── Cell 10: Visualise Predictions vs Ground Truth ──────────────────
from src.validate import extract_predictions

# Get predictions for a validation event
val_ds = ds.filter_by_model("1")
val_sample = None
for i in range(len(val_ds)):
    s = val_ds[i]
    if s["event_id"] == "4":
        val_sample = s
        break

if val_sample is not None:
    val_graph = build_unified_graph(val_sample)
    pred_data = extract_predictions(trainer.model, val_graph, DEVICE)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f"Predictions vs Ground Truth — Model {val_sample['model_id']}, "
                 f"Event {val_sample['event_id']}", fontsize=13)

    skip = pred_data["spinup_steps"]

    # 1D: first 4 nodes
    n_show = min(4, pred_data["preds_1d"].shape[1])
    for i in range(min(2, n_show)):
        ax = axes[0, i]
        ax.plot(pred_data["targets_1d"][:, i].numpy(), 'b-', label="Ground Truth", alpha=0.8)
        ax.plot(pred_data["preds_1d"][:, i].numpy(), 'r--', label="Prediction", alpha=0.8)
        ax.axvline(x=skip, color='gray', linestyle=':', label="Spinup end")
        ax.set_title(f"1D Node {i}")
        ax.set_xlabel("Timestep")
        ax.set_ylabel("Water Level (ft)")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    # 2D: sample nodes
    n_2d = pred_data["preds_2d"].shape[1]
    sample_2d_nodes = [0, n_2d // 4] if n_2d > 1 else [0]
    for i, node_idx in enumerate(sample_2d_nodes[:2]):
        ax = axes[1, i]
        ax.plot(pred_data["targets_2d"][:, node_idx].numpy(), 'b-', label="Ground Truth", alpha=0.8)
        ax.plot(pred_data["preds_2d"][:, node_idx].numpy(), 'r--', label="Prediction", alpha=0.8)
        ax.axvline(x=skip, color='gray', linestyle=':', label="Spinup end")
        ax.set_title(f"2D Node {node_idx}")
        ax.set_xlabel("Timestep")
        ax.set_ylabel("Water Level (ft)")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("logs/prediction_vs_truth.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Validation event not found.")

## 7. Submission Generation

Generate the competition submission CSV from the best trained model.

In [ ]:
# ── Cell 11: Generate Submission ────────────────────────────────────
from src.inference import SubmissionGenerator

# Use the best-trained model
gen = SubmissionGenerator(
    model=trainer.model,
    data_root=str(RAW_DATA_PATH),
    device=DEVICE,
    spinup_steps=10,
    verbose=True,
)

submission_df = gen.generate()

# Preview
print(f"\nSubmission preview:")
print(submission_df.head(20))
print(f"\nShape: {submission_df.shape}")
print(f"Unique models: {submission_df['model_id'].unique()}")
print(f"Unique events: {sorted(submission_df['event_id'].unique())}")
print(f"Node types: {submission_df['node_type'].unique()}")

# Save
gen.save(submission_df, "submission.csv")

---

## Scaling Up: Full Training Configuration

Once the pipeline is verified, use this config for the real training run:

In [ ]:
# ── Cell 12: Production Training Config (DO NOT RUN IN NOTEBOOK) ───
# Copy this to a .py script or CLI command for real training.
# Running 60+ epochs in a notebook is not recommended.

production_cfg = {
    "data_root": str(RAW_DATA_PATH),
    "model_ids": ["1", "2"],
    "val_event_id": "4",
    
    "hidden_channels": 128,
    "num_gnn_layers": 3,
    "num_gru_layers": 2,
    "dropout": 0.1,
    
    "epochs": 60,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "grad_clip_norm": 1.0,
    
    "tf_warmup_epochs": 10,
    "tf_decay_epochs": 30,
    "tf_min_ratio": 0.0,
    
    "pushforward_K": 10,
    "use_push_forward": True,
    "temporal_scheme": "linear",
    
    "scheduler": "cosine",
    "early_stop_patience": 15,
    
    "use_amp": True,
    "device": "auto",
}

print("Production training config:")
print(json.dumps(production_cfg, indent=2))
print("\nTo run: python -m src.trainer --epochs 60 --hidden_channels 128")